# 🌙 LunaFemCare — Phase 1: Data Processing & ML Model Training

**Architecture Overview:**
- **Model 1 – Nutrient Processor**: TF-IDF + Cosine Similarity food-item matcher. Parses natural-language food input (e.g. "100g Poha") and returns a 1-D nutrient vector `[Energy, Protein, Fat, Carbs, Fiber, Iron, Calcium, Vit_A, Sodium]`.
- **Model 2 – Recommendation & Risk Engine**: Rule-based inference engine. Inputs are Model 1's nutrient vector + contextual variables `[Phase, Is_Pregnant, Symptoms]`. Computes gap against personalised baseline and outputs actionable food suggestions with deficiency flags.

**Datasets used:**
- `nutrients_data.csv` — 1 070 Indian food items with macros/micros per serving
- `female_nutritionˍinfoset.csv` — 500 female user profiles with personalised daily requirements

---

## 0 · Environment Setup

In [ ]:
# Install dependencies (run once)
import subprocess, sys
packages = ['pandas', 'numpy', 'scikit-learn', 'joblib', 'matplotlib', 'seaborn']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])
print('✅ All packages ready.')

In [ ]:
import pandas as pd
import numpy as np
import re, os, json, warnings
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR   = BASE_DIR
MODEL_DIR  = os.path.join(os.getcwd(), 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

NUTRIENTS_CSV = os.path.join(DATA_DIR, 'nutrients_data.csv')
FEMALE_CSV    = os.path.join(DATA_DIR, 'female_nutrition\u02cdinfoset.csv')

print(f'BASE_DIR  : {BASE_DIR}')
print(f'MODEL_DIR : {MODEL_DIR}')
print(f'Nutrients CSV exists : {os.path.exists(NUTRIENTS_CSV)}')
print(f'Female CSV exists    : {os.path.exists(FEMALE_CSV)}')

---
## 1 · Data Engineering — `nutrients_data.csv`

In [ ]:
# ── Load ───────────────────────────────────────────────────────────────────────
raw_nutrients = pd.read_csv(NUTRIENTS_CSV)
print(f'Shape  : {raw_nutrients.shape}')
print(f'Columns: {list(raw_nutrients.columns)}')
raw_nutrients.head(5)

In [ ]:
# ── Parse Serving_Size → numeric grams / ml ────────────────────────────────────
def parse_serving_size(s: str) -> float:
    """Convert '100g', '150ml', '200g' → float grams/ml."""
    if pd.isna(s):
        return 100.0
    match = re.search(r'([\d\.]+)', str(s))
    return float(match.group(1)) if match else 100.0

df_food = raw_nutrients.copy()
df_food['Serving_g'] = df_food['Serving_Size'].apply(parse_serving_size)

# ── Normalise all nutrient columns to per-100 g ─────────────────────────────────
NUTRIENT_COLS = ['Energy_kcal','Protein_g','Fat_g','Carbs_g','Fiber_g',
                 'Iron_mg','Calcium_mg','Vit_A_mcg','Sodium_mg']

for col in NUTRIENT_COLS:
    df_food[col + '_per100g'] = df_food[col] / df_food['Serving_g'] * 100

NORM_COLS = [c + '_per100g' for c in NUTRIENT_COLS]

# ── Verify no NaNs ─────────────────────────────────────────────────────────────
print('Missing values after normalisation:')
print(df_food[NORM_COLS].isnull().sum())
df_food[['Food_Name','Category','Serving_g'] + NORM_COLS].head(8)

In [ ]:
# ── Build a richer text corpus for TF-IDF matching ─────────────────────────────
# We lowercase and clean the food name, prepend category for richer context
df_food['text_corpus'] = (
    df_food['Food_Name'].str.lower().str.replace(r'[^a-z\s]', ' ', regex=True)
    + ' ' +
    df_food['Category'].str.lower().str.replace(r'[^a-z\s]', ' ', regex=True)
)

print('Sample corpus entries:')
print(df_food['text_corpus'].head(10).to_string())

In [ ]:
# ── EDA: Distribution of Energy per 100 g ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df_food['Energy_kcal_per100g'], bins=40, color='#7c4d9a', edgecolor='white', alpha=0.85)
axes[0].set_title('Energy (kcal/100 g)', fontsize=13); axes[0].set_xlabel('kcal')

axes[1].hist(df_food['Iron_mg_per100g'], bins=40, color='#c0392b', edgecolor='white', alpha=0.85)
axes[1].set_title('Iron (mg/100 g)', fontsize=13); axes[1].set_xlabel('mg')

axes[2].hist(df_food['Protein_g_per100g'], bins=40, color='#27ae60', edgecolor='white', alpha=0.85)
axes[2].set_title('Protein (g/100 g)', fontsize=13); axes[2].set_xlabel('g')

plt.suptitle('LunaFemCare · Nutrient Distribution (per 100 g)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'nutrient_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Nutrient distribution chart saved.')

In [ ]:
# ── Category summary ──────────────────────────────────────────────────────────
cat_summary = df_food.groupby('Category')[NORM_COLS].mean().round(2)
print(f'Unique categories: {df_food["Category"].nunique()}')
cat_summary

---
## 2 · Data Engineering — `female_nutritionˍinfoset.csv`

In [ ]:
# ── Load ───────────────────────────────────────────────────────────────────────
raw_female = pd.read_csv(FEMALE_CSV)
print(f'Shape  : {raw_female.shape}')
print(f'Columns: {list(raw_female.columns)}')
raw_female.head(5)

In [ ]:
# ── Check value distributions for key categorical columns ─────────────────────
print('Is_Pregnant  :', raw_female['Is_Pregnant'].value_counts().to_dict())
print('Health_Status:', raw_female['Health_Status'].value_counts().to_dict())
print('Activity     :', raw_female['Activity_Level'].value_counts().to_dict())
print('Stress       :', raw_female['Stress_Level'].value_counts().to_dict())

In [ ]:
# ── Pregnancy logic: pregnant users get higher iron + folate thresholds ────────
# Non-pregnant baseline iron = 18 mg/day (already in data)
# Pregnant baseline iron     = 27 mg/day (as per ICMR)
# The `female_nutritionˍinfoset.csv` already encodes 27 mg for pregnant rows

df_female = raw_female.copy()

# Encode boolean
df_female['Is_Pregnant_flag'] = (df_female['Is_Pregnant'] == 'Yes').astype(int)

# Verify pregnancy-based iron threshold enforcement
assert df_female.loc[df_female['Is_Pregnant_flag'] == 1, 'Required_Iron_mg'].min() == 27, \
    'Pregnant iron requirement should be 27 mg!'
assert df_female.loc[df_female['Is_Pregnant_flag'] == 0, 'Required_Iron_mg'].min() == 18, \
    'Non-pregnant iron requirement should be 18 mg!'

print('✅ Pregnancy iron threshold validation passed.')
print(f"Pregnant users    : {df_female['Is_Pregnant_flag'].sum()}")
print(f"Non-pregnant users: {(df_female['Is_Pregnant_flag'] == 0).sum()}")

In [ ]:
# ── Compute population-level average baselines (used as fallback) ───────────────
BASELINE_COLS = [
    'Required_Energy_kcal','Required_Protein_g','Required_Fat_g',
    'Required_Carbs_g','Required_Fiber_g','Required_Iron_mg',
    'Required_Calcium_mg','Required_Vit_A_mcg','Required_Sodium_mg'
]

# Segment-level averages (non-pregnant vs pregnant)
baseline_normal   = df_female[df_female['Is_Pregnant_flag'] == 0][BASELINE_COLS].mean().round(1)
baseline_pregnant = df_female[df_female['Is_Pregnant_flag'] == 1][BASELINE_COLS].mean().round(1)

print('=== Average baseline — Non-pregnant ===')
print(baseline_normal)
print('\n=== Average baseline — Pregnant ===')
print(baseline_pregnant)

In [ ]:
# ── EDA: BMI distribution and health status breakdown ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_female['BMI'], bins=30, color='#e67e22', edgecolor='white', alpha=0.85)
axes[0].set_title('BMI Distribution', fontsize=13)
axes[0].axvline(18.5, color='#3498db', linestyle='--', label='Underweight')
axes[0].axvline(25.0, color='#2ecc71', linestyle='--', label='Normal|Overweight')
axes[0].axvline(30.0, color='#e74c3c', linestyle='--', label='Obese')
axes[0].legend(fontsize=8)

status_counts = df_female['Health_Status'].value_counts()
colors = ['#9b59b6','#3498db','#e74c3c','#2ecc71','#f39c12']
axes[1].bar(status_counts.index, status_counts.values, color=colors[:len(status_counts)], edgecolor='white')
axes[1].set_title('Health Status Distribution', fontsize=13)
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('LunaFemCare · Female User Profile EDA', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'female_profile_eda.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 3 · Model 1 — Nutrient Processor (TF-IDF + Cosine Similarity)

**Goal**: Given a free-text input such as `"100g Poha"` or `"2 pieces Idli"`, return a nutrient vector scaled to the actual portion consumed.


In [ ]:
# ── Step 3.1: Fit TF-IDF Vectoriser on food corpus ─────────────────────────────
tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),   # character n-grams handle typos / partial matches
    min_df=1,
    max_features=8000,
    sublinear_tf=True
)

food_matrix = tfidf.fit_transform(df_food['text_corpus'])
print(f'TF-IDF matrix shape: {food_matrix.shape}   (items × features)')
print(f'Vocab size: {len(tfidf.vocabulary_)}')

In [ ]:
# ── Step 3.2: Quantity Extractor ────────────────────────────────────────────────
UNIT_TO_GRAMS = {
    'g': 1.0, 'gram': 1.0, 'grams': 1.0,
    'kg': 1000.0, 'kilogram': 1000.0,
    'ml': 1.0, 'milliliter': 1.0, 'millilitre': 1.0,
    'l': 1000.0, 'liter': 1000.0, 'litre': 1000.0,
    'cup': 240.0, 'cups': 240.0,
    'tbsp': 15.0, 'tablespoon': 15.0, 'tablespoons': 15.0,
    'tsp': 5.0, 'teaspoon': 5.0, 'teaspoons': 5.0,
    'bowl': 200.0, 'bowls': 200.0,
    'plate': 250.0, 'plates': 250.0,
    'piece': 100.0, 'pieces': 100.0,
    'packet': 100.0,
    'serving': 100.0, 'servings': 100.0,
}

QUANTITY_PATTERN = re.compile(
    r'(?P<qty>[\d\.]+)\s*(?P<unit>' + '|'.join(UNIT_TO_GRAMS.keys()) + r')',
    re.IGNORECASE
)

def extract_quantity_grams(text: str) -> float:
    """Return the quantity in grams extracted from a user text string."""
    match = QUANTITY_PATTERN.search(text)
    if match:
        qty  = float(match.group('qty'))
        unit = match.group('unit').lower()
        return qty * UNIT_TO_GRAMS.get(unit, 1.0)
    # Try plain number alone (assume grams)
    num_match = re.search(r'([\d\.]+)', text)
    if num_match:
        return float(num_match.group(1))
    return 100.0  # default 100 g

# Quick tests
tests = ['100g Poha', '2 pieces Idli', '1 bowl DalTadka', '150ml Chicken Soup', '3 tbsp ghee']
for t in tests:
    print(f'{t!r:35s} → {extract_quantity_grams(t):>8.1f} g')

In [ ]:
# ── Step 3.3: Main Nutrient Processor function ─────────────────────────────────
def get_nutrient_vector(food_text: str, top_k: int = 1) -> dict:
    """
    Parse `food_text` (e.g. '100g Poha') and return a nutrient dict
    scaled to the specified quantity.

    Returns
    -------
    dict with keys: matched_food, similarity_score, quantity_g, nutrients (sub-dict)
    """
    if not food_text or not food_text.strip():
        return {'error': 'Empty input'}

    quantity_g = extract_quantity_grams(food_text)

    # Clean query: strip digits, units → keep food name tokens
    query_clean = re.sub(r'[\d\.]+\s*(' + '|'.join(UNIT_TO_GRAMS.keys()) + r')', '', food_text, flags=re.I)
    query_clean = re.sub(r'[^a-z\s]', ' ', query_clean.lower()).strip()

    if not query_clean:
        query_clean = food_text.lower()

    query_vec = tfidf.transform([query_clean])
    sims      = cosine_similarity(query_vec, food_matrix).flatten()
    top_idx   = int(np.argmax(sims))
    score     = float(sims[top_idx])

    row = df_food.iloc[top_idx]

    # Scale from per-100g to actual quantity
    scale = quantity_g / 100.0

    label_map = {
        'Energy_kcal': 'Energy_kcal_per100g',
        'Protein_g'  : 'Protein_g_per100g',
        'Fat_g'      : 'Fat_g_per100g',
        'Carbs_g'    : 'Carbs_g_per100g',
        'Fiber_g'    : 'Fiber_g_per100g',
        'Iron_mg'    : 'Iron_mg_per100g',
        'Calcium_mg' : 'Calcium_mg_per100g',
        'Vit_A_mcg'  : 'Vit_A_mcg_per100g',
        'Sodium_mg'  : 'Sodium_mg_per100g',
    }

    nutrients = {k: round(row[v] * scale, 3) for k, v in label_map.items()}

    return {
        'matched_food'    : row['Food_Name'],
        'category'        : row['Category'],
        'similarity_score': round(score, 4),
        'quantity_g'      : round(quantity_g, 1),
        'nutrients'       : nutrients,
    }


# ── Test the Nutrient Processor ────────────────────────────────────────────────
test_inputs = [
    '100g Poha',
    '2 pieces Idli',
    '150g Palak Paneer',
    '1 bowl Dal Makhani',
    '1 cup chicken curry',
    '200ml Chicken Soup',
    '50g Roasted Chana',
    '100g Gajar Halwa',
    '1 plate Chole Bhature',
    '75g Ragi Dosa',
]

print('=' * 70)
print(f'{"Input Text":<30} {"Matched":<22} {"Qty(g)":>6} {"Score":>6} {"Energy":>7} {"Iron":>6}')
print('=' * 70)
results_store = []
for inp in test_inputs:
    r = get_nutrient_vector(inp)
    results_store.append(r)
    print(f"{inp:<30} {r['matched_food']:<22} {r['quantity_g']:>6.1f} {r['similarity_score']:>6.3f} "
          f"{r['nutrients']['Energy_kcal']:>7.1f} {r['nutrients']['Iron_mg']:>6.2f}")
print('=' * 70)

In [ ]:
# ── Multi-item meal aggregator ─────────────────────────────────────────────────
def aggregate_meal(food_list: list) -> dict:
    """
    Sum nutrient vectors for a list of food strings.
    food_list = ['100g Poha', '1 piece Idli', ...]
    """
    totals = {k: 0.0 for k in ['Energy_kcal','Protein_g','Fat_g','Carbs_g',
                                'Fiber_g','Iron_mg','Calcium_mg','Vit_A_mcg','Sodium_mg']}
    items  = []
    for food in food_list:
        r = get_nutrient_vector(food)
        if 'error' not in r:
            items.append({'input': food, 'matched': r['matched_food'], 'qty_g': r['quantity_g']})
            for k in totals:
                totals[k] = round(totals[k] + r['nutrients'][k], 3)
    return {'items': items, 'total_nutrients': totals}

# Example: a typical lunch
lunch = ['150g VegBiryani', '1 bowl RajmaMasala', '200g CurdRice']
meal_result = aggregate_meal(lunch)

print('Meal Aggregation Test:')
print(json.dumps(meal_result, indent=2))

---
## 4 · Model 2 — Recommendation & Risk Engine (Rule-Based Inference)

**Inputs**: 
- `nutrient_vector` (total nutrients consumed so far today – from Model 1)
- `context` dict with keys: `phase` (Follicular / Ovulatory / Luteal / Menstrual), `is_pregnant` (bool), `symptoms` (list), `user_baseline` (personalised daily requirements)

**Outputs**: 
- Deficiency flags (`iron_low`, `calcium_low`, ...)
- Top food suggestions per deficiency
- Phase-specific wellness tips
- Risk score (0–100)

In [ ]:
# ── Step 4.1: Phase-based nutrient priority multipliers ───────────────────────
# Each menstrual phase shifts how urgently certain nutrients are prioritised.
# Multiplier > 1 means the phase increases importance of that nutrient.

PHASE_MULTIPLIERS = {
    'Menstrual': {
        'Iron_mg'    : 1.5,   # heavy blood loss → prioritise iron strongly
        'Calcium_mg' : 1.2,   # reduces cramps
        'Fiber_g'    : 1.1,   # digestive regulation
        'Energy_kcal': 1.05,
        'Protein_g'  : 1.0,
        'Vit_A_mcg'  : 1.0,
        'Fat_g'      : 1.0,
        'Carbs_g'    : 1.0,
        'Sodium_mg'  : 1.0,
    },
    'Follicular': {
        'Iron_mg'    : 1.1,
        'Protein_g'  : 1.2,   # muscle rebuilding + rising estrogen
        'Energy_kcal': 1.0,
        'Calcium_mg' : 1.0,
        'Fiber_g'    : 1.1,
        'Fat_g'      : 1.0,
        'Carbs_g'    : 1.1,
        'Vit_A_mcg'  : 1.1,
        'Sodium_mg'  : 1.0,
    },
    'Ovulatory': {
        'Protein_g'  : 1.15,
        'Fiber_g'    : 1.2,
        'Iron_mg'    : 1.0,
        'Calcium_mg' : 1.0,
        'Energy_kcal': 1.05,
        'Fat_g'      : 1.0,
        'Carbs_g'    : 1.0,
        'Vit_A_mcg'  : 1.2,
        'Sodium_mg'  : 1.0,
    },
    'Luteal': {
        'Calcium_mg' : 1.3,   # reduces PMS mood swings
        'Fiber_g'    : 1.3,   # bloating management
        'Iron_mg'    : 1.0,
        'Protein_g'  : 1.1,
        'Energy_kcal': 1.1,   # cravings are real!
        'Fat_g'      : 1.1,
        'Carbs_g'    : 1.15,
        'Vit_A_mcg'  : 1.0,
        'Sodium_mg'  : 0.9,   # reduces water retention
    },
    'Pregnancy': {             # overrides phase entirely
        'Iron_mg'    : 2.0,   # 27 mg requirement
        'Calcium_mg' : 1.5,   # foetal bone development
        'Protein_g'  : 1.5,
        'Energy_kcal': 1.3,
        'Fiber_g'    : 1.2,
        'Vit_A_mcg'  : 1.3,
        'Fat_g'      : 1.1,
        'Carbs_g'    : 1.1,
        'Sodium_mg'  : 1.0,
    },
}

print('Phase multiplier dictionary loaded. Phases:', list(PHASE_MULTIPLIERS.keys()))

In [ ]:
# ── Step 4.2: Symptom → Nutrient flag mapping ─────────────────────────────────
SYMPTOM_NUTRIENT_MAP = {
    'fatigue'     : ['Iron_mg', 'Energy_kcal', 'Protein_g'],
    'cramps'      : ['Calcium_mg', 'Fiber_g'],
    'bloating'    : ['Fiber_g', 'Sodium_mg'],
    'mood_swings' : ['Calcium_mg', 'Carbs_g'],
    'acne'        : ['Vit_A_mcg', 'Fiber_g'],
    'headache'    : ['Iron_mg', 'Sodium_mg'],
    'nausea'      : ['Protein_g', 'Carbs_g'],
    'backache'    : ['Calcium_mg', 'Iron_mg'],
    'insomnia'    : ['Calcium_mg', 'Protein_g'],
    'heavy_flow'  : ['Iron_mg', 'Protein_g'],
}

print('Symptom map loaded for', len(SYMPTOM_NUTRIENT_MAP), 'symptoms.')

In [ ]:
# ── Step 4.3: Food suggestion index — for each nutrient, top foods ─────────────
# Pre-compute top-20 foods for each key micronutrient (per 100 g)

TOP_FOODS_PER_NUTRIENT = {}

for nutrient, col in [
    ('Iron_mg',    'Iron_mg_per100g'),
    ('Calcium_mg', 'Calcium_mg_per100g'),
    ('Protein_g',  'Protein_g_per100g'),
    ('Fiber_g',    'Fiber_g_per100g'),
    ('Vit_A_mcg',  'Vit_A_mcg_per100g'),
    ('Energy_kcal','Energy_kcal_per100g'),
    ('Carbs_g',    'Carbs_g_per100g'),
]:
    top = (
        df_food[['Food_Name', 'Category', col]]
        .sort_values(col, ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
    TOP_FOODS_PER_NUTRIENT[nutrient] = top.rename(columns={col: 'value_per100g'})

# Preview
print('Top 5 Iron-rich foods:')
print(TOP_FOODS_PER_NUTRIENT['Iron_mg'].head(5).to_string())
print('\nTop 5 Calcium-rich foods:')
print(TOP_FOODS_PER_NUTRIENT['Calcium_mg'].head(5).to_string())

In [ ]:
# ── Step 4.4: Core Recommendation Engine function ──────────────────────────────

def recommend(
    nutrient_consumed: dict,
    context: dict,
    n_suggestions: int = 5,
) -> dict:
    """
    Model 2 — Recommendation & Risk Engine.

    Parameters
    ----------
    nutrient_consumed : dict  (keys match NUTRIENT_COLS names)
        Cumulative nutrients consumed so far today (from Model 1 output).
    context : dict
        {
          'phase'         : str   (Menstrual / Follicular / Ovulatory / Luteal),
          'is_pregnant'   : bool,
          'symptoms'      : list of str,
          'user_baseline' : dict  (personalised daily requirements)
        }
    n_suggestions : int
        Number of food recommendations per deficient nutrient.

    Returns
    -------
    dict with deficiency analysis, food suggestions, phase tips, risk_score.
    """
    is_pregnant  = context.get('is_pregnant', False)
    phase        = 'Pregnancy' if is_pregnant else context.get('phase', 'Follicular')
    symptoms     = [s.lower().replace(' ', '_') for s in context.get('symptoms', [])]
    user_baseline = context.get('user_baseline', {})

    # ── Resolve phase multipliers ──────────────────────────────────────────────
    mults = PHASE_MULTIPLIERS.get(phase, PHASE_MULTIPLIERS['Follicular'])

    # ── Effective daily requirements = baseline × phase multiplier ─────────────
    effective_reqs = {
        'Energy_kcal': user_baseline.get('Required_Energy_kcal', 2000) * mults.get('Energy_kcal', 1.0),
        'Protein_g'  : user_baseline.get('Required_Protein_g',   60)   * mults.get('Protein_g',   1.0),
        'Fat_g'      : user_baseline.get('Required_Fat_g',       55)   * mults.get('Fat_g',       1.0),
        'Carbs_g'    : user_baseline.get('Required_Carbs_g',     280)  * mults.get('Carbs_g',     1.0),
        'Fiber_g'    : user_baseline.get('Required_Fiber_g',     25)   * mults.get('Fiber_g',     1.0),
        'Iron_mg'    : user_baseline.get('Required_Iron_mg',     18)   * mults.get('Iron_mg',     1.0),
        'Calcium_mg' : user_baseline.get('Required_Calcium_mg',  1000) * mults.get('Calcium_mg',  1.0),
        'Vit_A_mcg'  : user_baseline.get('Required_Vit_A_mcg',  700)  * mults.get('Vit_A_mcg',  1.0),
        'Sodium_mg'  : user_baseline.get('Required_Sodium_mg',  2000) * mults.get('Sodium_mg',   1.0),
    }

    # ── Gap analysis ──────────────────────────────────────────────────────────
    gaps      = {}
    pct_met   = {}
    deficient = []

    for nut, req in effective_reqs.items():
        consumed = nutrient_consumed.get(nut, 0.0)
        gap      = req - consumed
        pct      = round(consumed / req * 100, 1) if req > 0 else 100.0
        gaps[nut]    = round(gap, 2)
        pct_met[nut] = pct
        if pct < 75:          # < 75% of daily req → flagged as deficient
            deficient.append(nut)

    # ── Boost deficiency list based on reported symptoms ──────────────────────
    symptom_flagged = []
    for sym in symptoms:
        nuts_from_sym = SYMPTOM_NUTRIENT_MAP.get(sym, [])
        for nut in nuts_from_sym:
            if nut not in symptom_flagged:
                symptom_flagged.append(nut)

    # Merge: deficient ∪ symptom-flagged (de-duped)
    priority_nutrients = list(dict.fromkeys(deficient + symptom_flagged))

    # ── Build food suggestions ────────────────────────────────────────────────
    suggestions = {}
    for nut in priority_nutrients:
        if nut in TOP_FOODS_PER_NUTRIENT:
            top_foods = TOP_FOODS_PER_NUTRIENT[nut].head(n_suggestions)
            suggestions[nut] = [
                {
                    'food'       : row['Food_Name'],
                    'category'   : row['Category'],
                    'per_100g'   : round(row['value_per100g'], 2),
                    'gap_needed' : gaps.get(nut, 0),
                }
                for _, row in top_foods.iterrows()
            ]

    # ── Phase-specific wellness tips ──────────────────────────────────────────
    PHASE_TIPS = {
        'Menstrual'  : [
            'Increase iron intake from spinach, ragi, and sesame seeds to replenish blood loss.',
            'Warm herbal teas (ginger, chamomile) can ease cramps.',
            'Avoid excess salt to reduce bloating.',
            'Gentle yoga and rest are recommended; reduce high-intensity exercise.',
        ],
        'Follicular' : [
            'Rising estrogen supports energy — great time for more complex carbs.',
            'Include fermented foods (idli, dosa, curd) to boost gut health.',
            'Protein-rich foods support muscle repair after menstruation.',
            'This is your most energetic phase — ideal for strength training.',
        ],
        'Ovulatory'  : [
            'Peak energy and metabolism — fuel with balanced meals.',
            'Cruciferous vegetables help liver process excess estrogen.',
            'Vitamin A-rich foods (carrot, spinach) support cell health.',
            'Stay well hydrated — cervical fluid increases now.',
        ],
        'Luteal'     : [
            'Calcium from dairy or sesame seeds helps manage mood swings and PMS.',
            'High-fiber foods (oats, dal, vegetables) reduce bloating.',
            'Complex carbs stabilise serotonin levels.',
            'Limit caffeine and alcohol to prevent cramps and disrupted sleep.',
        ],
        'Pregnancy'  : [
            'Your iron requirement is 27 mg/day — include fortified foods, spinach, and lentils.',
            'Folic acid (from leafy greens) is critical for neural tube development.',
            'Calcium supports foetal bone growth — have dairy, ragi, or sesame daily.',
            'Small, frequent meals help manage nausea and maintain steady blood sugar.',
            'Avoid raw papaya, pineapple, and excess Vitamin A supplements.',
        ],
    }

    tips = PHASE_TIPS.get(phase, [])

    # ── Risk score (0-100) ────────────────────────────────────────────────────
    # Based on (1) number of deficient nutrients + (2) symptom burden
    n_deficient     = len(deficient)
    n_symptoms      = len(symptoms)
    pregnancy_bonus = 20 if is_pregnant else 0  # higher accountability for pregnancy

    avg_deficit_pct = np.mean(
        [max(0, 100 - pct_met[n]) for n in deficient]
    ) if deficient else 0

    risk_score = min(100, round(
        avg_deficit_pct * 0.5
        + n_symptoms * 5
        + pregnancy_bonus, 1
    ))

    return {
        'phase'              : phase,
        'is_pregnant'        : is_pregnant,
        'effective_daily_req': {k: round(v, 1) for k, v in effective_reqs.items()},
        'pct_met'            : pct_met,
        'gaps'               : gaps,
        'deficient_nutrients': deficient,
        'symptom_flagged'    : symptom_flagged,
        'food_suggestions'   : suggestions,
        'wellness_tips'      : tips,
        'risk_score'         : risk_score,
    }


print('✅ Recommendation engine function defined.')

In [ ]:
# ── Test Scenario 1: Non-pregnant, Menstrual phase, fatigue + cramps ───────────
consumed_today = aggregate_meal([
    '100g Poha',
    '1 bowl Dal Tadka',
    '150g Curd Rice',
])['total_nutrients']

user_baseline_example = {
    'Required_Energy_kcal': 2000, 'Required_Protein_g': 60, 'Required_Fat_g': 55,
    'Required_Carbs_g': 280, 'Required_Fiber_g': 25, 'Required_Iron_mg': 18,
    'Required_Calcium_mg': 1000, 'Required_Vit_A_mcg': 700, 'Required_Sodium_mg': 2000,
}

rec1 = recommend(
    nutrient_consumed=consumed_today,
    context={
        'phase'        : 'Menstrual',
        'is_pregnant'  : False,
        'symptoms'     : ['fatigue', 'cramps'],
        'user_baseline': user_baseline_example,
    }
)

print('=== TEST SCENARIO 1: Menstrual | Fatigue + Cramps ===')
print(f"Phase       : {rec1['phase']}")
print(f"Risk Score  : {rec1['risk_score']} / 100")
print(f"Deficient   : {rec1['deficient_nutrients']}")
print(f"Sym-flagged : {rec1['symptom_flagged']}")
print(f"% Met Iron  : {rec1['pct_met']['Iron_mg']}%")
print(f"% Met Ca    : {rec1['pct_met']['Calcium_mg']}%")
print('')
print('Top Iron suggestions:')
for s in rec1['food_suggestions'].get('Iron_mg', [])[:3]:
    print(f"  • {s['food']} ({s['category']}) — {s['per_100g']} mg/100g")
print('\nWellness tips:')
for tip in rec1['wellness_tips']:
    print(f'  → {tip}')

In [ ]:
# ── Test Scenario 2: Pregnant user with nausea ──────────────────────────────────
pregnant_baseline = {
    'Required_Energy_kcal': 2500, 'Required_Protein_g': 71, 'Required_Fat_g': 70,
    'Required_Carbs_g': 340, 'Required_Fiber_g': 28, 'Required_Iron_mg': 27,
    'Required_Calcium_mg': 1000, 'Required_Vit_A_mcg': 770, 'Required_Sodium_mg': 2000,
}

consumed_preg = aggregate_meal([
    '100g Upma',
    '100g OatsIdli',
])['total_nutrients']

rec2 = recommend(
    nutrient_consumed=consumed_preg,
    context={
        'phase'        : 'Follicular',    # overridden by is_pregnant
        'is_pregnant'  : True,
        'symptoms'     : ['nausea', 'fatigue'],
        'user_baseline': pregnant_baseline,
    }
)

print('=== TEST SCENARIO 2: Pregnant | Nausea + Fatigue ===')
print(f"Phase       : {rec2['phase']}   (overridden to Pregnancy)")
print(f"Risk Score  : {rec2['risk_score']} / 100")
print(f"Deficient   : {rec2['deficient_nutrients']}")
print(f"Effective Iron req : {rec2['effective_daily_req']['Iron_mg']} mg")
print('')
print('Top Protein suggestions (for nausea):')
for s in rec2['food_suggestions'].get('Protein_g', [])[:3]:
    print(f"  • {s['food']} ({s['category']}) — {s['per_100g']} g/100g")
print('\nPregnancy wellness tips:')
for tip in rec2['wellness_tips']:
    print(f'  → {tip}')

In [ ]:
# ── Test Scenario 3: Luteal phase, mood_swings + bloating ─────────────────────
consumed_luteal = aggregate_meal([
    '200g ChickenBiryani',
    '1 piece MasalaDosa',
])['total_nutrients']

rec3 = recommend(
    nutrient_consumed=consumed_luteal,
    context={
        'phase'        : 'Luteal',
        'is_pregnant'  : False,
        'symptoms'     : ['mood_swings', 'bloating'],
        'user_baseline': user_baseline_example,
    }
)

print('=== TEST SCENARIO 3: Luteal | Mood Swings + Bloating ===')
print(f"Phase       : {rec3['phase']}")
print(f"Risk Score  : {rec3['risk_score']} / 100")
print(f"Deficient   : {rec3['deficient_nutrients']}")
print(f"% Met Fiber : {rec3['pct_met']['Fiber_g']}%")
print(f"% Met Ca    : {rec3['pct_met']['Calcium_mg']}%")

---
## 5 · Validation — 10 Distinct Test Inputs End-to-End

In [ ]:
TEST_SCENARIOS = [
    {'meal': ['100g Poha', '1 bowl DalTadka'],        'phase': 'Menstrual',  'pregnant': False, 'symptoms': ['fatigue']},
    {'meal': ['2 pieces Idli', '1 bowl SambarRice'],  'phase': 'Follicular', 'pregnant': False, 'symptoms': []},
    {'meal': ['150g PalakPaneer', '200g CurdRice'],   'phase': 'Ovulatory',  'pregnant': False, 'symptoms': ['acne']},
    {'meal': ['100g VegBiryani', '1 bowl RajmaMasala'],'phase': 'Luteal',   'pregnant': False, 'symptoms': ['bloating', 'cramps']},
    {'meal': ['100g Upma', '50g RoastedChana'],       'phase': 'Menstrual',  'pregnant': False, 'symptoms': ['heavy_flow', 'fatigue']},
    {'meal': ['100g MasalaDosa', '1 bowl DalMakhani'], 'phase': 'Follicular','pregnant': True,  'symptoms': ['nausea']},
    {'meal': ['200g ChickenBiryani'],                  'phase': 'Ovulatory', 'pregnant': False, 'symptoms': []},
    {'meal': ['100g RagiDosa', '100g VegetableDalia'], 'phase': 'Luteal',    'pregnant': False, 'symptoms': ['insomnia', 'mood_swings']},
    {'meal': ['150g PaneerButterMasala', '200g VegPulav'],'phase': 'Menstrual','pregnant': False,'symptoms': ['backache']},
    {'meal': ['100g OatsUpma', '100g SproutedSalad'],  'phase': 'Follicular', 'pregnant': True, 'symptoms': ['fatigue', 'nausea']},
]

print(f'{'#':<4} {'Phase':<12} {'Preg':<6} {'Symptoms':<30} {'Risk':>5} {'Deficient Nutrients'}')
print('-' * 95)

all_results = []
for i, sc in enumerate(TEST_SCENARIOS, 1):
    meal_result = aggregate_meal(sc['meal'])
    rec = recommend(
        nutrient_consumed=meal_result['total_nutrients'],
        context={
            'phase'        : sc['phase'],
            'is_pregnant'  : sc['pregnant'],
            'symptoms'     : sc['symptoms'],
            'user_baseline': user_baseline_example,
        }
    )
    all_results.append(rec)
    deficits = ', '.join(rec['deficient_nutrients'][:4]) or 'None'
    syms     = ', '.join(sc['symptoms'])[:28] or 'None'
    print(f"{i:<4} {rec['phase']:<12} {'Yes' if sc['pregnant'] else 'No':<6} {syms:<30} {rec['risk_score']:>5} {deficits}")

print('\n✅ All 10 end-to-end scenarios passed.')

In [ ]:
# ── Validate that risk_score is always 0-100 and deficient is a list ──────────
for i, r in enumerate(all_results, 1):
    assert 0 <= r['risk_score'] <= 100,    f'Scenario {i}: risk_score out of range'
    assert isinstance(r['deficient_nutrients'], list), f'Scenario {i}: deficient not a list'
    assert isinstance(r['food_suggestions'], dict),    f'Scenario {i}: suggestions not a dict'
    assert r['phase'] in PHASE_MULTIPLIERS,            f'Scenario {i}: unknown phase'

print('✅ All assertion checks passed for all 10 scenarios.')

In [ ]:
# ── Risk Score Visualisation ──────────────────────────────────────────────────
phases     = [r['phase'] for r in all_results]
scores     = [r['risk_score'] for r in all_results]
pregnancies = ['🤰' if r['is_pregnant'] else '' for r in all_results]
labels     = [f"S{i+1} {pregnancies[i]}" for i in range(len(all_results))]

phase_colors = {
    'Menstrual' : '#e74c3c',
    'Follicular': '#3498db',
    'Ovulatory' : '#2ecc71',
    'Luteal'    : '#9b59b6',
    'Pregnancy' : '#f39c12',
}
bar_colors = [phase_colors.get(p, '#95a5a6') for p in phases]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(labels, scores, color=bar_colors, edgecolor='white', linewidth=0.8, width=0.6)
ax.axhline(50, color='#e74c3c', linestyle='--', linewidth=1, alpha=0.7, label='High risk threshold')
ax.axhline(25, color='#f39c12', linestyle=':', linewidth=1, alpha=0.7, label='Moderate risk threshold')

for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            str(score), ha='center', va='bottom', fontsize=9, fontweight='bold')

# Legend for phases
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=p) for p, c in phase_colors.items()]
ax.legend(handles=legend_elements + [
    plt.Line2D([0], [0], color='#e74c3c', linestyle='--', label='High risk threshold'),
    plt.Line2D([0], [0], color='#f39c12', linestyle=':', label='Moderate risk threshold'),
], loc='upper right', fontsize=8)

ax.set_ylim(0, 110)
ax.set_xlabel('Test Scenario', fontsize=12)
ax.set_ylabel('Risk Score', fontsize=12)
ax.set_title('LunaFemCare · Model 2 Risk Score — 10 Test Scenarios', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'risk_score_validation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Risk score chart saved.')

---
## 6 · Serialisation — Export Pipeline Artefacts

All artefacts are saved to `ml/models/` for backend consumption.

In [ ]:
# ── 6.1: Save TF-IDF vectoriser ────────────────────────────────────────────────
tfidf_path = os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl')
joblib.dump(tfidf, tfidf_path)
print(f'✅ TF-IDF vectoriser saved  → {tfidf_path}')

# ── 6.2: Save food matrix (TF-IDF transformed) ────────────────────────────────
import scipy.sparse as sp
matrix_path = os.path.join(MODEL_DIR, 'food_tfidf_matrix.pkl')
joblib.dump(food_matrix, matrix_path)
print(f'✅ Food TF-IDF matrix saved  → {matrix_path}')

# ── 6.3: Save cleaned food DataFrame as Parquet (fast read) ───────────────────
food_parquet_path = os.path.join(MODEL_DIR, 'food_database.parquet')
df_food.to_parquet(food_parquet_path, index=False)
print(f'✅ Food database (parquet) saved → {food_parquet_path}')

# ── 6.4: Save phase multipliers & configs as JSON ─────────────────────────────
config = {
    'version'             : '1.0.0',
    'norm_cols'           : NORM_COLS,
    'nutrient_cols'       : NUTRIENT_COLS,
    'phase_multipliers'   : PHASE_MULTIPLIERS,
    'symptom_nutrient_map': SYMPTOM_NUTRIENT_MAP,
    'unit_to_grams'       : UNIT_TO_GRAMS,
    'baseline_normal'     : baseline_normal.to_dict(),
    'baseline_pregnant'   : baseline_pregnant.to_dict(),
    'deficiency_threshold': 75,   # % of daily req below which a nutrient is flagged
    'top_k_suggestions'   : 5,
}

config_path = os.path.join(MODEL_DIR, 'luna_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'✅ Config JSON saved          → {config_path}')

# ── 6.5: Save top foods per nutrient index ────────────────────────────────────
top_foods_serialisable = {
    k: v.to_dict(orient='records') for k, v in TOP_FOODS_PER_NUTRIENT.items()
}
top_foods_path = os.path.join(MODEL_DIR, 'top_foods_index.json')
with open(top_foods_path, 'w') as f:
    json.dump(top_foods_serialisable, f, indent=2)
print(f'✅ Top-foods index saved       → {top_foods_path}')

# ── 6.6: Save full recommendation engine pipeline as pkl ─────────────────────
pipeline = {
    'tfidf'                  : tfidf,
    'food_matrix'            : food_matrix,
    'df_food'                : df_food,
    'phase_multipliers'      : PHASE_MULTIPLIERS,
    'symptom_nutrient_map'   : SYMPTOM_NUTRIENT_MAP,
    'top_foods_per_nutrient' : TOP_FOODS_PER_NUTRIENT,
    'unit_to_grams'          : UNIT_TO_GRAMS,
    'baseline_normal'        : baseline_normal.to_dict(),
    'baseline_pregnant'      : baseline_pregnant.to_dict(),
}
pipeline_path = os.path.join(MODEL_DIR, 'luna_pipeline.pkl')
joblib.dump(pipeline, pipeline_path, compress=3)
print(f'✅ Full pipeline pkl saved    → {pipeline_path}')

In [ ]:
# ── 6.7: Verify all artefact files exist ──────────────────────────────────────
expected_files = [
    'tfidf_vectorizer.pkl',
    'food_tfidf_matrix.pkl',
    'food_database.parquet',
    'luna_config.json',
    'top_foods_index.json',
    'luna_pipeline.pkl',
    'nutrient_distribution.png',
    'female_profile_eda.png',
    'risk_score_validation.png',
]

all_ok = True
print('Artefact verification:')
for fname in expected_files:
    path = os.path.join(MODEL_DIR, fname)
    size = os.path.getsize(path) if os.path.exists(path) else -1
    icon = '✅' if size > 0 else '❌'
    if size < 0:
        all_ok = False
    print(f'  {icon} {fname:<40} {size:>10,} bytes')

print(f'\n{"✅ ALL ARTEFACTS VERIFIED SUCCESSFULLY" if all_ok else "❌ SOME ARTEFACTS ARE MISSING"}')

---
## 7 · Reload & Smoke-Test (Simulating Backend Deserialization)

Verify that the saved pipeline can be loaded from disk and produce identical outputs — mimicking what Phase 2's FastAPI backend will do.

In [ ]:
# ── Load pipeline from disk ────────────────────────────────────────────────────
loaded = joblib.load(pipeline_path)

# Re-assign for use in this scope
_tfidf     = loaded['tfidf']
_matrix    = loaded['food_matrix']
_df_food   = loaded['df_food']
_top_foods = loaded['top_foods_per_nutrient']
_symp_map  = loaded['symptom_nutrient_map']
_phases    = loaded['phase_multipliers']
_units     = loaded['unit_to_grams']

print('Pipeline loaded from disk.')
print(f'  TF-IDF vocab size : {len(_tfidf.vocabulary_)}')
print(f'  Food DB rows      : {len(_df_food)}')
print(f'  Phase keys        : {list(_phases.keys())}')

# Quick smoke test on a new query
q    = _tfidf.transform(['palak paneer main course'])
sims = cosine_similarity(q, _matrix).flatten()
best = _df_food.iloc[int(np.argmax(sims))]['Food_Name']
print(f'\nSMOKE TEST — query: "palak paneer" → matched: {best}')
assert 'Paneer' in best or 'palak' in best.lower() or sims.max() > 0.1, 'Smoke test failed!'
print('✅ Reload smoke test passed.')

---
## 8 · Phase 1 Summary

| Artefact | Description |
|---|---|
| `tfidf_vectorizer.pkl` | Fitted TF-IDF vectoriser (char n-gram, 2–4) |
| `food_tfidf_matrix.pkl` | Pre-transformed food corpus matrix |
| `food_database.parquet` | Cleaned food DB normalised to per-100 g |
| `luna_pipeline.pkl` | Full serialised pipeline (model 1 + model 2 data) |
| `luna_config.json` | Phase multipliers, symptom map, threshold config |
| `top_foods_index.json` | Pre-computed top-20 foods per micronutrient |
| `nutrient_distribution.png` | EDA chart — nutrient distributions |
| `female_profile_eda.png` | EDA chart — BMI & health status |
| `risk_score_validation.png` | Validation chart — 10 test scenarios |

### ✅ Model Capabilities
- **Model 1 (Nutrient Processor)**: Parses free-text Indian food queries with quantity extraction (grams, ml, pieces, bowls, etc.) and returns scaled nutrient vectors via TF-IDF cosine similarity.
- **Model 2 (Recommendation Engine)**: Computes phase-adjusted nutrient gaps against personalised baselines, flags deficiencies from symptom inputs, suggests top phase-appropriate foods, and produces a 0–100 risk score.
- **Pregnancy Logic**: When `is_pregnant=True`, phase flags are nullified and iron requirement is boosted to 27 mg/day; all multipliers shift to the `Pregnancy` profile.

### 🔜 Next Steps — Phase 2
Load `luna_pipeline.pkl` in a **FastAPI** backend (`backend/main.py`) and expose:
- `POST /api/process_food` → calls Model 1
- `POST /api/get_recommendations` → calls Model 2
